# Module 02: Exploring the Power Platform Connector Catalog via API

## Learning Objectives

By completing this notebook you will:

1. Authenticate to the Microsoft Power Platform API using client credentials
2. Retrieve the full connector catalog for a Power Platform environment
3. Inspect connector metadata: tier (standard/premium), authentication type, available actions and triggers
4. Compare standard vs premium connectors programmatically using pandas
5. Filter and search connectors to answer real architectural questions

## Prerequisites

- Azure Active Directory app registration with `https://api.powerplatform.com/.default` permission granted
- Tenant ID, Client ID, and Client Secret from the app registration
- Python packages: `requests`, `pandas`, `python-dotenv`
- A Power Platform environment ID (found in Power Automate portal: Settings → Session details)

## Estimated Time: 15 minutes

---

### Setup Instructions

Before running this notebook:

1. Create an Azure AD app registration at [portal.azure.com](https://portal.azure.com) → Azure Active Directory → App registrations → New registration
2. Under **Certificates & secrets**, create a new client secret. Copy the value — it is shown only once.
3. Under **API permissions**, add `https://api.powerplatform.com/user_impersonation` (or use client credentials with admin consent)
4. Copy your **Tenant ID** (Azure AD → Overview) and **Application (client) ID**
5. Create a `.env` file in this directory with:

```
AZURE_TENANT_ID=your-tenant-id-here
AZURE_CLIENT_ID=your-client-id-here
AZURE_CLIENT_SECRET=your-client-secret-here
POWER_PLATFORM_ENVIRONMENT_ID=your-environment-id-here
```

In [ ]:
# Install required packages if not already present
# Run this cell once, then restart the kernel
import subprocess, sys

required = ["requests", "pandas", "python-dotenv", "tabulate"]
for pkg in required:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

print("All packages installed.")

In [ ]:
# Core imports
# requests: HTTP calls to the Power Platform and Azure AD APIs
# pandas: tabular analysis of connector metadata
# dotenv: load credentials from a .env file (keeps secrets out of notebook cells)
import os
import json
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file
# If .env does not exist, the os.getenv calls below will return None
# and the authentication step will raise an informative error
load_dotenv()

TENANT_ID      = os.getenv("AZURE_TENANT_ID")
CLIENT_ID      = os.getenv("AZURE_CLIENT_ID")
CLIENT_SECRET  = os.getenv("AZURE_CLIENT_SECRET")
ENVIRONMENT_ID = os.getenv("POWER_PLATFORM_ENVIRONMENT_ID")

# Validate that all required credentials are present
missing = [name for name, val in [
    ("AZURE_TENANT_ID", TENANT_ID),
    ("AZURE_CLIENT_ID", CLIENT_ID),
    ("AZURE_CLIENT_SECRET", CLIENT_SECRET),
    ("POWER_PLATFORM_ENVIRONMENT_ID", ENVIRONMENT_ID),
] if not val]

if missing:
    raise EnvironmentError(
        f"Missing required environment variables: {missing}\n"
        "Create a .env file in this directory with the required values.\n"
        "See the Setup Instructions in the first cell."
    )

print("Credentials loaded successfully.")
print(f"Tenant ID:      {TENANT_ID[:8]}...")
print(f"Client ID:      {CLIENT_ID[:8]}...")
print(f"Environment ID: {ENVIRONMENT_ID}")

## 1. Authenticate: Obtain an Access Token

The Power Platform API uses Azure Active Directory OAuth 2.0 client credentials flow.
The flow works like this:

```
This script                          Azure AD token endpoint
     │                                       │
     │── POST /oauth2/v2.0/token ───────────►│
     │   client_id, client_secret, scope     │
     │◄── access_token (JWT, 1 hour) ────────│
     │
     │── GET /connectors (Bearer token) ────►│  Power Platform API
     │◄── connector list JSON ───────────────│
```

The access token is a JSON Web Token (JWT) that expires after 1 hour.
For scripts that run longer, implement token refresh logic.
For this notebook, one token is sufficient.

In [ ]:
def get_access_token(tenant_id: str, client_id: str, client_secret: str) -> str:
    """
    Obtain an OAuth 2.0 access token from Azure AD using client credentials flow.

    Parameters
    ----------
    tenant_id : str
        Azure AD tenant GUID (found in Azure portal → Azure Active Directory → Overview)
    client_id : str
        Application (client) ID of the registered app
    client_secret : str
        Client secret value (shown once at creation time)

    Returns
    -------
    str
        Bearer token string to include in Authorization headers

    Raises
    ------
    requests.HTTPError
        If Azure AD returns a non-200 response (wrong credentials, revoked secret, etc.)
    """
    token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"

    # The scope tells Azure AD which service this token is for.
    # .default means "grant all permissions pre-consented for this app registration"
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "scope":         "https://api.powerplatform.com/.default",
    }

    response = requests.post(token_url, data=payload, timeout=30)
    response.raise_for_status()  # Raise HTTPError for 4xx/5xx responses

    token_data = response.json()
    return token_data["access_token"]


# Obtain the token
access_token = get_access_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)

# Show only the first 40 chars — never print a full token to shared output
print(f"Access token obtained: {access_token[:40]}...")
print("Token is valid for approximately 1 hour.")

## 2. List All Available Connectors

The Power Platform Connectors API endpoint returns metadata for every connector available
in the specified environment. The response is paginated — we follow `nextLink` URLs
until we have retrieved the complete catalog.

API reference: `GET https://api.powerplatform.com/connectivity/connectors?api-version=1`

Each connector object in the response contains:
- `id`: unique connector identifier path
- `name`: internal name (e.g. `shared_office365`)
- `properties.displayName`: human-readable name (e.g. `Office 365 Outlook`)
- `properties.tier`: `Standard` or `Premium`
- `properties.connectionParameters`: authentication configuration
- `properties.capabilities`: list of strings (`actions`, `triggers`)
- `properties.publisher`: the organisation that built the connector

In [ ]:
def list_all_connectors(access_token: str, environment_id: str) -> list[dict]:
    """
    Retrieve the complete connector catalog for a Power Platform environment.

    Follows pagination automatically — the API returns pages of up to 100 connectors
    with a `nextLink` URL in the response body when more pages exist.

    Parameters
    ----------
    access_token : str
        Valid Bearer token for the Power Platform API scope
    environment_id : str
        Power Platform environment GUID

    Returns
    -------
    list[dict]
        List of raw connector objects from the API response
    """
    base_url = "https://api.powerplatform.com/connectivity/connectors"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept":        "application/json",
    }
    params = {
        "api-version": "1",
        "$filter":     f"environment eq '{environment_id}'",
    }

    all_connectors = []
    url = base_url
    page = 0

    while url:
        # For subsequent pages, params are already embedded in the nextLink URL
        if page == 0:
            response = requests.get(url, headers=headers, params=params, timeout=30)
        else:
            response = requests.get(url, headers=headers, timeout=30)

        response.raise_for_status()
        data = response.json()

        page_connectors = data.get("value", [])
        all_connectors.extend(page_connectors)

        # Follow pagination: nextLink is present only when more pages exist
        url = data.get("nextLink")
        page += 1

        print(f"  Page {page}: retrieved {len(page_connectors)} connectors "
              f"(total so far: {len(all_connectors)})")

    return all_connectors


print("Fetching connector catalog...")
raw_connectors = list_all_connectors(access_token, ENVIRONMENT_ID)
print(f"\nTotal connectors retrieved: {len(raw_connectors)}")

## 3. Parse Connector Metadata into a DataFrame

The raw API response is deeply nested JSON. We extract the fields most useful
for analysis into a flat pandas DataFrame.

Fields we extract:

| Field | Description |
|-------|-------------|
| `name` | Internal connector ID |
| `display_name` | Human-readable name |
| `tier` | `Standard` or `Premium` |
| `publisher` | Who built the connector |
| `has_actions` | Boolean — connector has at least one action |
| `has_triggers` | Boolean — connector has at least one trigger |
| `auth_type` | Primary authentication method |
| `description` | Short description from the connector definition |

In [ ]:
def extract_auth_type(connection_parameters: dict) -> str:
    """
    Extract the primary authentication type from connector connectionParameters.

    The connectionParameters object describes what credentials the connector
    needs. The type field on the first parameter indicates the auth mechanism.

    Common values returned:
    - 'oauthSetting'   → OAuth 2.0
    - 'securestring'   → API key or Basic auth
    - 'string'         → Connection string or URL
    - 'bool'           → Boolean configuration flag
    - ''               → No authentication (public connectors)
    """
    if not connection_parameters:
        return "none"

    # connectionParameters is a dict of parameter_name → parameter_definition
    # We examine the 'type' field of each parameter to identify auth mechanism
    types_seen = set()
    for param_def in connection_parameters.values():
        if isinstance(param_def, dict):
            param_type = param_def.get("type", "")
            types_seen.add(param_type)

    # Priority: OAuth takes precedence over other types
    if "oauthSetting" in types_seen:
        return "oauth2"
    if "securestring" in types_seen:
        return "api_key_or_basic"
    if "string" in types_seen:
        return "connection_string"
    return "other"


def parse_connectors(raw_connectors: list[dict]) -> pd.DataFrame:
    """
    Transform the raw connector API response into an analysis-ready DataFrame.

    Parameters
    ----------
    raw_connectors : list[dict]
        Raw connector objects from list_all_connectors()

    Returns
    -------
    pd.DataFrame
        One row per connector with flattened metadata columns
    """
    rows = []

    for connector in raw_connectors:
        props = connector.get("properties", {})
        capabilities = props.get("capabilities", [])
        conn_params   = props.get("connectionParameters", {})

        rows.append({
            "name":         connector.get("name", ""),
            "display_name": props.get("displayName", ""),
            "tier":         props.get("tier", "Standard"),
            "publisher":    props.get("publisher", ""),
            "has_actions":  "actions"  in capabilities,
            "has_triggers": "triggers" in capabilities,
            "auth_type":    extract_auth_type(conn_params),
            "description":  props.get("description", "")[:120],  # Truncate for display
        })

    df = pd.DataFrame(rows)

    # Normalise tier casing — API sometimes returns 'standard' in lowercase
    df["tier"] = df["tier"].str.capitalize()

    return df


connectors_df = parse_connectors(raw_connectors)

print(f"DataFrame shape: {connectors_df.shape[0]} connectors × {connectors_df.shape[1]} columns")
print(f"\nColumns: {list(connectors_df.columns)}")
connectors_df.head(5)

## 4. Standard vs Premium Breakdown

Let's count connectors by tier and visualise the split.
This answers the licensing question: how many premium connectors are available
compared to standard, and what authentication types does each tier use?

In [ ]:
# Count connectors by tier
# value_counts() returns a Series sorted by frequency descending
tier_counts = connectors_df["tier"].value_counts()

print("=== Connectors by Tier ===")
for tier, count in tier_counts.items():
    pct = count / len(connectors_df) * 100
    bar = "█" * int(pct / 2)
    print(f"  {tier:12s}: {count:4d}  {bar} ({pct:.1f}%)")

print()

# Break down auth types within each tier
# This shows how auth method distribution differs between standard and premium
print("=== Auth Type by Tier ===")
auth_by_tier = (
    connectors_df
    .groupby(["tier", "auth_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["tier", "count"], ascending=[True, False])
)
print(auth_by_tier.to_string(index=False))

In [ ]:
# Connectors that have triggers (can start flows)
# vs connectors that only have actions (can only be used in flow body)

trigger_summary = pd.DataFrame({
    "Total connectors":        [len(connectors_df)],
    "Has triggers":            [connectors_df["has_triggers"].sum()],
    "Has actions":             [connectors_df["has_actions"].sum()],
    "Has both":                [(connectors_df["has_triggers"] & connectors_df["has_actions"]).sum()],
    "Actions only (no trig)": [(connectors_df["has_actions"] & ~connectors_df["has_triggers"]).sum()],
})

print("=== Connector Capabilities ===")
print(trigger_summary.T.to_string(header=False))

print("\nNote: most connectors support both triggers and actions.")
print("Action-only connectors include utility connectors like Data Operations and Variables.")

## 5. Inspect a Specific Connector's Full Metadata

The connector list endpoint returns summary metadata. To get the full definition
of a specific connector — including every action, every trigger, and their
parameter schemas — we call the individual connector endpoint.

We will inspect the **Office 365 Outlook** connector in detail.

In [ ]:
def get_connector_details(access_token: str, environment_id: str, connector_name: str) -> dict:
    """
    Retrieve the full definition of a single connector.

    The full definition includes:
    - All actions with their parameter schemas
    - All triggers with their output schemas
    - Authentication configuration details
    - API definitions (Swagger)

    Parameters
    ----------
    access_token : str
        Valid Bearer token
    environment_id : str
        Power Platform environment GUID
    connector_name : str
        Internal connector name (e.g. 'shared_office365')
        Obtain from the 'name' column of the connectors DataFrame

    Returns
    -------
    dict
        Full connector definition object
    """
    url = (
        f"https://api.powerplatform.com/connectivity/connectors/{connector_name}"
        f"?api-version=1&$filter=environment eq '{environment_id}'"
    )
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept":        "application/json",
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()


# Find the internal name for Office 365 Outlook in our DataFrame
# The display name might have slight variations — search case-insensitively
outlook_rows = connectors_df[
    connectors_df["display_name"].str.contains("Office 365 Outlook", case=False, na=False)
]

if outlook_rows.empty:
    print("Office 365 Outlook connector not found in catalog.")
    print("Available Outlook-related connectors:")
    print(connectors_df[connectors_df["display_name"].str.contains("Outlook", case=False, na=False)][["name", "display_name"]])
else:
    outlook_connector_name = outlook_rows.iloc[0]["name"]
    print(f"Found: {outlook_rows.iloc[0]['display_name']}")
    print(f"Internal name: {outlook_connector_name}")

    outlook_details = get_connector_details(access_token, ENVIRONMENT_ID, outlook_connector_name)
    print("\nFull connector details retrieved.")

In [ ]:
# Extract and display all actions from the Outlook connector
# Actions are defined in the Swagger/OpenAPI spec embedded in the connector definition

def extract_operations(connector_details: dict, operation_type: str = "actions") -> pd.DataFrame:
    """
    Extract all actions or triggers from a connector's full definition.

    The connector definition embeds a Swagger spec under
    properties.apiDefinitions.originalSwaggerUrl (or inline).
    Operations are listed under properties.swagger.paths[path][method].

    Parameters
    ----------
    connector_details : dict
        Full connector definition from get_connector_details()
    operation_type : str
        'actions' or 'triggers' — filters by x-ms-visibility and trigger metadata

    Returns
    -------
    pd.DataFrame
        One row per operation with name, summary, and parameter count
    """
    props = connector_details.get("properties", {})

    # The swagger definition may be inline or referenced — check both locations
    swagger = props.get("swagger", {})
    if not swagger:
        swagger = props.get("apiDefinitions", {}).get("swagger", {})

    paths = swagger.get("paths", {})
    rows = []

    for path, path_item in paths.items():
        for method, operation in path_item.items():
            if not isinstance(operation, dict):
                continue

            extensions = operation.get("x-ms-api-annotation", {})
            is_trigger = "x-ms-trigger" in operation

            # Filter by operation type requested
            if operation_type == "triggers" and not is_trigger:
                continue
            if operation_type == "actions" and is_trigger:
                continue

            params = operation.get("parameters", [])
            rows.append({
                "operation_id":  operation.get("operationId", ""),
                "summary":       operation.get("summary", ""),
                "description":   operation.get("description", "")[:100],
                "method":        method.upper(),
                "path":          path,
                "param_count":   len(params),
                "visibility":    operation.get("x-ms-visibility", "important"),
            })

    return pd.DataFrame(rows)


# Display Outlook actions
if 'outlook_details' in dir():
    outlook_actions = extract_operations(outlook_details, "actions")
    print(f"Office 365 Outlook — Actions ({len(outlook_actions)} total)")
    print("=" * 60)
    if not outlook_actions.empty:
        display_cols = ["operation_id", "summary", "param_count"]
        available_cols = [c for c in display_cols if c in outlook_actions.columns]
        print(outlook_actions[available_cols].to_string(index=False))
    else:
        print("No actions found in Swagger definition (API may use a reference URL).")
        print("Falling back to capabilities metadata:")
        print(json.dumps(outlook_details.get("properties", {}).get("capabilities", []), indent=2))

In [ ]:
# Display Outlook triggers
if 'outlook_details' in dir():
    outlook_triggers = extract_operations(outlook_details, "triggers")
    print(f"Office 365 Outlook — Triggers ({len(outlook_triggers)} total)")
    print("=" * 60)
    if not outlook_triggers.empty:
        display_cols = ["operation_id", "summary", "param_count"]
        available_cols = [c for c in display_cols if c in outlook_triggers.columns]
        print(outlook_triggers[available_cols].to_string(index=False))
    else:
        print("Trigger definitions embedded in Swagger paths not found.")
        print("Check properties.capabilities for trigger availability confirmation.")

## 6. Search and Filter the Connector Catalog

With the full catalog in a DataFrame, we can answer architectural questions
that would take minutes of manual browsing in the portal.

Work through each query below. After running each cell, read the output
and connect it to what you learned in the connectors guide.

In [ ]:
# Query 1: Which connectors use OAuth 2.0 authentication and are Standard tier?
# These are the connectors your organisation can use without a premium license
# and that use the most secure, user-delegated authentication.

standard_oauth = connectors_df[
    (connectors_df["tier"] == "Standard") &
    (connectors_df["auth_type"] == "oauth2")
].sort_values("display_name")

print(f"Standard connectors using OAuth 2.0: {len(standard_oauth)}")
print()
print(standard_oauth[["display_name", "publisher", "has_triggers"]].
      rename(columns={"display_name": "Connector", "publisher": "Publisher", "has_triggers": "Has Triggers"}).
      to_string(index=False))

In [ ]:
# Query 2: Which Premium connectors have triggers (can start flows)?
# Premium triggers require a paid plan — knowing which ones exist
# helps scope license requirements before architecting a flow.

premium_with_triggers = connectors_df[
    (connectors_df["tier"] == "Premium") &
    (connectors_df["has_triggers"] == True)
].sort_values("display_name")

print(f"Premium connectors with triggers: {len(premium_with_triggers)}")
print()
print(premium_with_triggers[["display_name", "auth_type", "publisher"]].
      rename(columns={"display_name": "Connector", "auth_type": "Auth", "publisher": "Publisher"}).
      head(20).to_string(index=False))

if len(premium_with_triggers) > 20:
    print(f"  ... and {len(premium_with_triggers) - 20} more")

In [ ]:
# Query 3: Search for connectors by keyword in display name
# Simulates the portal search experience, programmatically

def search_connectors(df: pd.DataFrame, keyword: str) -> pd.DataFrame:
    """
    Search the connector catalog by keyword (case-insensitive) in display name or description.

    Parameters
    ----------
    df : pd.DataFrame
        Connector catalog DataFrame from parse_connectors()
    keyword : str
        Search term

    Returns
    -------
    pd.DataFrame
        Matching connectors sorted by tier then display name
    """
    mask = (
        df["display_name"].str.contains(keyword, case=False, na=False) |
        df["description"].str.contains(keyword, case=False, na=False)
    )
    return df[mask].sort_values(["tier", "display_name"])[
        ["display_name", "tier", "auth_type", "has_triggers", "has_actions"]
    ]


# Try different keywords to explore the catalog
for keyword in ["SQL", "Azure", "SAP", "email"]:
    results = search_connectors(connectors_df, keyword)
    print(f"Search '{keyword}': {len(results)} connectors")
    if not results.empty:
        print(results.head(5).to_string(index=False))
    print()

## Exercise: Connector Audit for a Hypothetical Flow

**Scenario:** Your organisation wants to build a flow that:
1. Triggers when a new item is added to a SharePoint list
2. Looks up customer details from an Azure SQL Database
3. Sends a personalised email via Outlook
4. Posts a summary to a Teams channel

**Task:** Using the `connectors_df` DataFrame, answer these questions:

1. Which connectors are needed? (name the specific connectors from the catalog)
2. What tier is each connector? (Standard or Premium?)
3. What is the minimum Power Automate license this flow requires?
4. How many connectors in the catalog support both triggers AND actions?

Write your answers as code using DataFrame operations below.
The assertions at the bottom validate your answers.

In [ ]:
# YOUR CODE HERE
# ---------------
# Use connectors_df to answer the four questions above.
# Assign your answers to the variables below.

# Question 1: List the display names of the four connectors needed
# Hint: search_connectors(connectors_df, "SharePoint") etc.
needed_connectors = []  # Replace with a list of display name strings

# Question 2: For each needed connector, what tier is it?
# Hint: filter connectors_df by display_name and check the tier column
connector_tiers = {}  # Replace with dict: {display_name: tier}

# Question 3: What is the minimum license required?
# Assign one of: 'Microsoft 365 (standard)' or 'Power Automate per-user plan'
minimum_license = ""  # Replace with your answer

# Question 4: How many connectors have both triggers and actions?
both_capabilities_count = 0  # Replace with the integer count

print("Needed connectors:", needed_connectors)
print("Connector tiers:",   connector_tiers)
print("Minimum license:",   minimum_license)
print("Both capabilities:", both_capabilities_count)

In [ ]:
# Self-check: run this cell to validate your answers

def validate_exercise(needed_connectors, connector_tiers, minimum_license, both_capabilities_count):
    errors = []

    # Check Q1: at least 4 connectors identified
    if len(needed_connectors) < 4:
        errors.append(
            f"Q1: Expected at least 4 connectors, got {len(needed_connectors)}. "
            "The flow needs SharePoint, SQL Server, Outlook, and Teams."
        )

    # Check Q2: at least one Premium tier identified
    if connector_tiers and "Premium" not in connector_tiers.values():
        errors.append(
            "Q2: SQL Server is a Premium connector — at least one connector should be Premium. "
            "Check: connectors_df[connectors_df['display_name'].str.contains('SQL Server')]['tier']"
        )

    # Check Q3: premium license required because SQL Server is premium
    if "per-user" not in minimum_license.lower() and "premium" not in minimum_license.lower():
        errors.append(
            "Q3: SQL Server is a Premium connector — the flow requires a Power Automate per-user plan "
            "(or per-flow plan). A standard M365 license is not sufficient."
        )

    # Check Q4: reasonable count (most connectors support both)
    expected_min = int(len(connectors_df) * 0.5)  # At least 50% should have both
    if both_capabilities_count < expected_min:
        errors.append(
            f"Q4: Expected at least {expected_min} connectors with both capabilities, "
            f"got {both_capabilities_count}. "
            "Use: (connectors_df['has_triggers'] & connectors_df['has_actions']).sum()"
        )

    if errors:
        print("Issues found:")
        for i, err in enumerate(errors, 1):
            print(f"  {i}. {err}")
    else:
        print("All checks passed.")
        print("You can correctly audit connector requirements from the Power Platform API.")


validate_exercise(needed_connectors, connector_tiers, minimum_license, both_capabilities_count)

## 7. Export the Connector Catalog

Save the full catalog to a CSV file. This is useful for:
- Sharing with stakeholders who need to understand licensing implications
- Offline reference when building flow architectures
- Tracking which connectors are available across environments

In [ ]:
# Export the full connector catalog to CSV
# Save to the resources directory of this module

output_path = Path("../resources/connector_catalog.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

connectors_df.to_csv(output_path, index=False)

print(f"Catalog saved to: {output_path.resolve()}")
print(f"Rows: {len(connectors_df)}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

# Print a final summary
print("\n=== Catalog Summary ===")
summary = connectors_df.groupby("tier").agg(
    total=("name", "count"),
    with_triggers=("has_triggers", "sum"),
    with_actions=("has_actions", "sum"),
    oauth_count=("auth_type", lambda x: (x == "oauth2").sum()),
).reset_index()
print(summary.to_string(index=False))

## Summary

### Key Takeaways

1. **The Power Platform API is queryable** — you do not have to browse the portal to understand the connector catalog. Programmatic access lets you audit, compare, and document connectors at scale.

2. **Tier determines license requirements** — a single premium connector in any flow requires all users of that flow to have a Power Automate per-user plan (or a per-flow plan to be assigned). Audit tier before committing to an architecture.

3. **Auth type tells you how to create the connection** — OAuth connectors (the majority of Microsoft connectors) use the signed-in user's identity. API key and connection string connectors need static credentials managed by your team.

4. **Full connector metadata lives in the Swagger spec** — every action and trigger, with its parameter schema, is embedded in the connector's OpenAPI definition. The `extract_operations()` function in this notebook is a starting point for deeper introspection.

### What's Next

Exercise 01 (`exercises/01_connector_matching.py`) puts this knowledge to work with scenario-based connector and trigger matching.

Module 03 picks up from here — once you know which connectors to use, you need expressions and data operations to transform the data flowing between them.

### Additional Resources

- [Power Platform Connectors API reference](https://learn.microsoft.com/en-us/rest/api/power-platform/connectivity)
- [Microsoft Graph Explorer](https://developer.microsoft.com/en-us/graph/graph-explorer) — interactive API explorer
- [Power Platform admin connector](https://learn.microsoft.com/en-us/connectors/powerplatformforadmins/) — flow management via flows